## Data Processing

In [32]:
import pandas as pd 
import os
from pathlib import Path
import sqlite3
import re

# Visualise the distribution of comments per post
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from datetime import datetime
from sqlalchemy import create_engine, text

# Import our custom Reddit API module

# --- Configuration for Jupyter ---
# The following magic command is for Jupyter notebooks to render plots inline.
# It should be commented out when running as a standalone script.
%config InlineBackend.figure_formats = ['svg']


In [33]:
project_root = Path.cwd().parent

player_file = project_root / "data" / "raw" / "player_stats.csv"
draft_file = project_root / "data" / "raw" / "draft_data.csv"
adp_file = project_root / "data" / "raw" / "FantasyPros_2024_Overall_ADP_Rankings.csv"

df_players = pd.read_csv(player_file)
df_draft = pd.read_csv(draft_file)
df_adp = pd.read_csv(adp_file, quotechar='"', escapechar='\\', on_bad_lines='skip')



In [34]:
df_players['games_played'] = df_players['actual_teamLoss'] + df_players['actual_teamWin']
df_players

,points,avg_points,projected_points,projected_avg_points,actual_rushingAttempts,actual_rushingYards,actual_rushingTouchdowns,actual_rushing2PtConversions,actual_27,actual_28,...,actual_defensive45PlusPointsAllowed,actual_196,actual_fumbleRecoveredForTD,actual_madeFieldGoalsFrom60Plus,actual_defensive2PtReturns,actual_puntFairCatches,actual_puntAverage38.0-39.9,proj_defensiveAssistedTackles,proj_defensiveSoloTackles,games_played
0,341.0,21.31,251.98,18.00,345.0,125.312500,13.0,3.0,395.0,191.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.0
1,286.0,16.82,210.02,15.00,203.0,53.352941,6.0,NaN,175.0,84.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.0
2,153.0,13.91,274.87,18.32,86.0,45.363636,6.0,1.0,96.0,45.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0
3,142.0,9.47,217.93,14.53,4.0,0.800000,NaN,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0
4,266.0,17.73,221.80,14.79,5.0,0.133333,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
519,0.0,0.00,12.58,0.84,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
520,3.0,0.43,57.03,3.80,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0
521,0.0,0.00,6.24,0.42,3.0,3.666667,NaN,NaN,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0
522,14.0,0.88,8.66,0.62,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.0


In [35]:
df_adp

,Rank,Player,Team,Bye,POS,ESPN,Sleeper,AVG
0,1.0,Christian McCaffrey,SF,9,RB1,1.0,1.0,1.0
1,2.0,CeeDee Lamb,DAL,7,WR1,3.0,3.0,3.0
2,3.0,Tyreek Hill,MIA,6,WR2,4.0,2.0,3.3
3,4.0,Bijan Robinson,ATL,12,RB2,2.0,6.0,4.3
4,5.0,Breece Hall,NYJ,12,RB3,5.0,8.0,5.0
...,...,...,...,...,...,...,...,...
573,576.0,Matt Ammendola,NaN,NaN,K48,498.0,NaN,498.0
574,577.0,Desmond Ridder,CIN,12,QB85,499.0,NaN,499.0
575,578.0,Michael Thomas,NaN,NaN,WR185,500.0,NaN,500.0
576,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Extract the position and posRank from the POS column

In [36]:
df_adp["position"] = df_adp["POS"].str.extract(r'([A-Za-z]+)')
df_adp["posRank"] = df_adp["POS"].str.extract(r'(\d+)').astype(float)
df_adp


,Rank,Player,Team,Bye,POS,ESPN,Sleeper,AVG,position,posRank
0,1.0,Christian McCaffrey,SF,9,RB1,1.0,1.0,1.0,RB,1.0
1,2.0,CeeDee Lamb,DAL,7,WR1,3.0,3.0,3.0,WR,1.0
2,3.0,Tyreek Hill,MIA,6,WR2,4.0,2.0,3.3,WR,2.0
3,4.0,Bijan Robinson,ATL,12,RB2,2.0,6.0,4.3,RB,2.0
4,5.0,Breece Hall,NYJ,12,RB3,5.0,8.0,5.0,RB,3.0
...,...,...,...,...,...,...,...,...,...,...
573,576.0,Matt Ammendola,NaN,NaN,K48,498.0,NaN,498.0,K,48.0
574,577.0,Desmond Ridder,CIN,12,QB85,499.0,NaN,499.0,QB,85.0
575,578.0,Michael Thomas,NaN,NaN,WR185,500.0,NaN,500.0,WR,185.0
576,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Match the player ids to the player

In [37]:
# Step 2: Map team names to ESPN-style D/ST names
team_to_dst = {
    "San Francisco 49ers": "49ers D/ST",
    "Chicago Bears": "Bears D/ST",
    "Cincinnati Bengals": "Bengals D/ST",
    "Buffalo Bills": "Bills D/ST",
    "Denver Broncos": "Broncos D/ST",
    "Cleveland Browns": "Browns D/ST",
    "Tampa Bay Buccaneers": "Buccaneers D/ST",
    "Arizona Cardinals": "Cardinals D/ST",
    "Los Angeles Chargers": "Chargers D/ST",
    "Kansas City Chiefs": "Chiefs D/ST",
    "Indianapolis Colts": "Colts D/ST",
    "Washington Commanders": "Commanders D/ST",
    "Dallas Cowboys": "Cowboys D/ST",
    "Miami Dolphins": "Dolphins D/ST",
    "Philadelphia Eagles": "Eagles D/ST",
    "Atlanta Falcons": "Falcons D/ST",
    "New York Giants": "Giants D/ST",
    "Jacksonville Jaguars": "Jaguars D/ST",
    "New York Jets": "Jets D/ST",
    "Detroit Lions": "Lions D/ST",
    "Green Bay Packers": "Packers D/ST",
    "Carolina Panthers": "Panthers D/ST",
    "New England Patriots": "Patriots D/ST",
    "Las Vegas Raiders": "Raiders D/ST",
    "Los Angeles Rams": "Rams D/ST",
    "Baltimore Ravens": "Ravens D/ST",
    "New Orleans Saints": "Saints D/ST",
    "Seattle Seahawks": "Seahawks D/ST",
    "Pittsburgh Steelers": "Steelers D/ST",
    "Houston Texans": "Texans D/ST",
    "Tennessee Titans": "Titans D/ST",
    "Minnesota Vikings": "Vikings D/ST",
}

# Step 3: Replace defense names in df_adp
df_adp['Player'] = df_adp['Player'].replace(team_to_dst)

In [38]:
import re

# Step 1: Define clean_name
def clean_name(name):
    if pd.isna(name):
        return ""
    # Remove suffixes like Jr., Sr., II, III, etc.
    name = re.sub(r'\b(jr\.?|sr\.?|ii|iii|iv|v)\b', '', name, flags=re.IGNORECASE)
    # Remove punctuation
    name = re.sub(r'[^\w\s]', '', name)
    # Normalize whitespace and lowercase
    return ' '.join(name.strip().lower().split())



# Step 4: Clean names
df_adp['Player_clean'] = df_adp['Player'].apply(clean_name)
df_players['player_name_clean'] = df_players['player_name'].apply(clean_name)

# Step 5: Create mapping and apply
player_id_map = dict(zip(df_players['player_name_clean'], df_players['player_id']))
df_adp['player_id'] = df_adp['Player_clean'].map(player_id_map)



In [39]:
df_adp

,Rank,Player,Team,Bye,POS,ESPN,Sleeper,AVG,position,posRank,Player_clean,player_id
0,1.0,Christian McCaffrey,SF,9,RB1,1.0,1.0,1.0,RB,1.0,christian mccaffrey,3117251.0
1,2.0,CeeDee Lamb,DAL,7,WR1,3.0,3.0,3.0,WR,1.0,ceedee lamb,4241389.0
2,3.0,Tyreek Hill,MIA,6,WR2,4.0,2.0,3.3,WR,2.0,tyreek hill,3116406.0
3,4.0,Bijan Robinson,ATL,12,RB2,2.0,6.0,4.3,RB,2.0,bijan robinson,4430807.0
4,5.0,Breece Hall,NYJ,12,RB3,5.0,8.0,5.0,RB,3.0,breece hall,4427366.0
...,...,...,...,...,...,...,...,...,...,...,...,...
573,576.0,Matt Ammendola,NaN,NaN,K48,498.0,NaN,498.0,K,48.0,matt ammendola,NaN
574,577.0,Desmond Ridder,CIN,12,QB85,499.0,NaN,499.0,QB,85.0,desmond ridder,4239086.0
575,578.0,Michael Thomas,NaN,NaN,WR185,500.0,NaN,500.0,WR,185.0,michael thomas,2976316.0
576,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN


In [40]:
float_to_int = ['Rank', 'ESPN', 'Sleeper', 'posRank', 'player_id']
df_adp['player_id'] = pd.to_numeric(df_adp['player_id'], errors='coerce').astype('Int64')


In [41]:
df_adp = df_adp.rename(columns={
    "Player": "player_name",
    "Team": "team_name",
    "AVG": "avg",
    # Rename columns to match conventions
})

In [42]:
df_draft.columns

Index(['autoDraftTypeId', 'id', 'lineupSlotId', 'memberId',
       'overallPickNumber', 'player_id', 'roundId', 'roundPickNumber',
       'team_id'],
      dtype='object')

In [43]:
list(df_players.columns)

['points',
 'avg_points',
 'projected_points',
 'projected_avg_points',
 'actual_rushingAttempts',
 'actual_rushingYards',
 'actual_rushingTouchdowns',
 'actual_rushing2PtConversions',
 'actual_27',
 'actual_28',
 'actual_29',
 'actual_30',
 'actual_31',
 'actual_32',
 'actual_33',
 'actual_34',
 'actual_rushing40PlusYardTD',
 'actual_rushing50PlusYardTD',
 'actual_rushing100To199YardGame',
 'actual_rushing200PlusYardGame',
 'actual_rushingYardsPerAttempt',
 'actual_receivingReceptions',
 'actual_receivingYards',
 'actual_receivingTouchdowns',
 'actual_47',
 'actual_48',
 'actual_49',
 'actual_50',
 'actual_51',
 'actual_receivingTargets',
 'actual_receivingYardsAfterCatch',
 'actual_receivingYardsPerReception',
 'actual_passingTimesSacked',
 'actual_65',
 'actual_66',
 'actual_fumbles',
 'actual_70',
 'actual_lostFumbles',
 'actual_turnovers',
 'actual_defensiveSoloTackles',
 'actual_defensiveTotalTackles',
 'actual_teamWin',
 'actual_teamLoss',
 'actual_pointsScored',
 'actual_179',


### Have to map the positions from the position_id slot to the position slot in the draft data

This allows us to fill in missing position data.

In [44]:
## RB = 2, WR = 4, WR = 23, QB = 20, TE = 6, K =17, D/ST = 16
position_map = {
    2: 'RB',
    4: 'WR', 
    23: 'WR',
    20: 'QB',
    0:'QB',
    6: 'TE',
    17: 'K',
    16: 'D/ST'
}
df = df_draft
df_draft['position'] = df_draft['lineupSlotId'].map(position_map)
df_draft

,autoDraftTypeId,id,lineupSlotId,memberId,overallPickNumber,player_id,roundId,roundPickNumber,team_id,position
0,0,1,2,{REDACTED-ESPN-MEMBER-ID},1,3117251,1,1,3,RB
1,0,2,4,{REDACTED-ESPN-MEMBER-ID},2,4241389,1,2,14,WR
2,0,3,2,{REDACTED-ESPN-MEMBER-ID},3,3929630,1,3,1,RB
3,0,4,0,{REDACTED-ESPN-MEMBER-ID},4,3918298,1,4,4,QB
4,3,5,2,NaN,5,4427366,1,5,8,RB
...,...,...,...,...,...,...,...,...,...,...
219,3,220,20,NaN,220,4048244,16,10,8,QB
220,1,221,20,NaN,221,4428209,16,11,4,QB
221,1,222,20,NaN,222,15965,16,12,1,QB
222,0,223,20,{REDACTED-ESPN-MEMBER-ID},223,4362887,16,13,14,QB


In [45]:
## Extract just the columns needed from the dataframe to prepare to 
## setup the database

player_columns = [
    "points",
    "avg_points",
    "projected_points",
    "projected_avg_points",
    "games_played",
    "actual_pointsScored",
    "player_name",
    "pro_team",
    "posRank",
    "player_id",
    "current_team_id",
    "current_team_name",
    "position"
]

adp_columns = [
    'player_name', 
    'team_name', 
    'ESPN', 
    'Sleeper',
    'posRank',
    'avg', 
    'position', 
    'player_id'
]

draft_columns = [
    'autoDraftTypeId', 
    'id', 
    'lineupSlotId',
    'overallPickNumber', 
    'player_id', 
    'roundId', 
    'roundPickNumber',
    'team_id'
]

In [46]:
player_database_df = df_players[player_columns]
adp_database_df = df_adp[adp_columns]
draft_database_df = df_draft[draft_columns]

In [47]:
df_draft

,autoDraftTypeId,id,lineupSlotId,memberId,overallPickNumber,player_id,roundId,roundPickNumber,team_id,position
0,0,1,2,{REDACTED-ESPN-MEMBER-ID},1,3117251,1,1,3,RB
1,0,2,4,{REDACTED-ESPN-MEMBER-ID},2,4241389,1,2,14,WR
2,0,3,2,{REDACTED-ESPN-MEMBER-ID},3,3929630,1,3,1,RB
3,0,4,0,{REDACTED-ESPN-MEMBER-ID},4,3918298,1,4,4,QB
4,3,5,2,NaN,5,4427366,1,5,8,RB
...,...,...,...,...,...,...,...,...,...,...
219,3,220,20,NaN,220,4048244,16,10,8,QB
220,1,221,20,NaN,221,4428209,16,11,4,QB
221,1,222,20,NaN,222,15965,16,12,1,QB
222,0,223,20,{REDACTED-ESPN-MEMBER-ID},223,4362887,16,13,14,QB


## DataBase Creation

Creating three tables.
    1. Draft_data
    2. Player_stats
    3. Average Draft Pick

In [48]:
engine = create_engine("sqlite:///../data/fantasy_data.db")

In [49]:
df_players

,points,avg_points,projected_points,projected_avg_points,actual_rushingAttempts,actual_rushingYards,actual_rushingTouchdowns,actual_rushing2PtConversions,actual_27,actual_28,...,actual_196,actual_fumbleRecoveredForTD,actual_madeFieldGoalsFrom60Plus,actual_defensive2PtReturns,actual_puntFairCatches,actual_puntAverage38.0-39.9,proj_defensiveAssistedTackles,proj_defensiveSoloTackles,games_played,player_name_clean
0,341.0,21.31,251.98,18.00,345.0,125.312500,13.0,3.0,395.0,191.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.0,saquon barkley
1,286.0,16.82,210.02,15.00,203.0,53.352941,6.0,NaN,175.0,84.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.0,devon achane
2,153.0,13.91,274.87,18.32,86.0,45.363636,6.0,1.0,96.0,45.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0,anthony richardson
3,142.0,9.47,217.93,14.53,4.0,0.800000,NaN,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,jaylen waddle
4,266.0,17.73,221.80,14.79,5.0,0.133333,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,malik nabers
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
519,0.0,0.00,12.58,0.84,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,brenden rice
520,3.0,0.43,57.03,3.80,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,trenton irwin
521,0.0,0.00,6.24,0.42,3.0,3.666667,NaN,NaN,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,velus jones
522,14.0,0.88,8.66,0.62,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.0,quintin morris


In [ ]:
# Define our database schema with specific data types
players_table_schema = """
CREATE TABLE players (
    player_id CHAR(7) PRIMARY KEY,
    player_name VARCHAR(100),
    current_team_name VARCHAR(100),
    posRank DECIMAL(4,2),
    current_team_id VARCHAR(10),
    position VARCHAR(20),
    pro_team VARCHAR(50),
    points DECIMAL(6,2),
    projected_points DECIMAL(6,2),
    avg_points DECIMAL(6,2) NOT NULL,
    projected_avg_points DECIMAL(6,2),
    actual_pointsScored DECIMAL(7,2),
    games_played INTEGER
)
"""

adp_schema = """
CREATE TABLE average_draft_position (
    adp_id INTEGER PRIMARY KEY AUTOINCREMENT,
    player_id CHAR(7),
    player_name VARCHAR(100),
    team_name VARCHAR(50),
    position VARCHAR(20),
    posRank INTEGER,
    espn INTEGER,
    sleeper INTEGER,
    avg FLOAT,
    FOREIGN KEY (player_id) REFERENCES players(player_id)
);
"""

draft_table_schema = """
CREATE TABLE draft (
    player_id CHAR(7) PRIMARY KEY,
    position VARCHAR(10),
    overallPickNumber INTEGER,
    team_id VARCHAR(4),
    roundPickNumber INTEGER,
    id INTEGER,
    roundId INTEGER,
    autoDraftTypeId INTEGER,
    lineupSlotId INTEGER,
    FOREIGN KEY (player_id) REFERENCES players(player_id)
);
"""

# Execute the schema creation
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS players;"))
    conn.execute(text("DROP TABLE IF EXISTS average_draft_position;"))
    conn.execute(text("DROP TABLE IF EXISTS draft;"))
    conn.execute(text(players_table_schema))
    conn.execute(text(adp_schema))
    conn.execute(text(draft_table_schema))
    conn.commit()

# Database tables created successfully
# - posts table with post_id as PRIMARY KEY
# - comments table with comment_id as PRIMARY KEY and post_id as FOREIGN KEY

In [51]:
player_database_df.to_sql('players', engine, if_exists='append', index=False)
adp_database_df.to_sql('average_draft_position', engine, if_exists='append', index=False)
draft_database_df.to_sql('draft', engine, if_exists='append', index=False)


print(f"✅ Database populated: ADP: {len(adp_database_df)}, Players: {len(player_database_df)}, Draft: {len(draft_database_df)}")

✅ Database populated: ADP: 578, Players: 524, Draft: 224
